# 06. California DWR carrot test supplement

[Notebook 01](01_external_reference_audit.ipynb) introduces the external sources and their roles. The 2023 California DWR survey adds a carrot-only external check in a different production setting.

Saved outputs may describe an earlier run. [Notebook 08](08_dataset_summary.ipynb) shows the parcels used in the saved experiment.

## 1. Settings for this run

Review the settings before running the notebook from top to bottom.

| Setting | What it controls |
|---|---|
| `run_satellite_collection` | Enables Earth Engine submission and retrieval, subject to the notebook's reference mode. |
| `submit_only` | Queues cloud work for retrieval in a later run. |
| `download_missing_references` | Allows the notebook to obtain the local reference snapshot if it is missing. |
| `parcels_per_batch` | Candidate parcel limit for each new batch. |
| `extra_batches` | Highest numbered extra slot to consider in the current round, after the regular batch. |

A **campaign** holds the collection goal and its records, a **round** groups batch plans, and a **batch** is a set of selected parcels with an imagery plan. Rerunning with the same extra-slot limit revisits the current round and can resume unfinished work. For example, changing 25 to 26 makes slot 0026 available in that round if eligible parcels remain. [Collection guide](README.md)

In [ ]:
# Edit these values before running collection.
run_satellite_collection = True
submit_only = False
download_missing_references = True
parcels_per_batch = 50
extra_batches = 1


In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup
from crop_pipeline.preprocessing_workflow import paths
from crop_pipeline.sources import adaptive, annual, eurocrops, supplemental

settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
_, report, _ = paths(root)
archive = annual.archive_root(settings, report, repository_root=root)
from crop_pipeline.notebook_display import show_table, show_mapping

## 2. Obtain references

Locate the local DWR single-crop carrot snapshot. With reference downloads enabled, the loader can obtain it if missing.

In [ ]:
references = root / "data/external/california_dwr/carrot_2023.geojson"
print("Reference file:", references)


## 3. Check parcel eligibility

Load the survey polygons with the T6 carrot label. These polygons supply field locations. The next step screens them for overlap before selecting a batch.

In [ ]:
features = supplemental.load_california(references, download_if_missing=download_missing_references)
show_mapping({"Carrot parcels before overlap screening": len(features)},
             title="California DWR reference pool")


## 4. Plan or resume batches

The planner uses eligible field footprints to create imagery requests. Saved plans carry their original selections and project splits. The batch table shows selected parcels and planned tiles for plans returned in this run.

In [ ]:
batches = supplemental.prepare(
    "california_dwr", report, archive, features, references,
    limit=parcels_per_batch, extra_batches=extra_batches,
)

# Count parcels and imagery tiles separately.
show_table([
    {"Batch": Path(context["folder"]).name, "Selected parcels": context["candidate_parcels"],
      "Planned tiles": plan["estimate"]["tiles"]}
    for plan, context in batches if plan is not None
], title="Parcel and imagery workload by batch")

## 5. Collect annual imagery

Earth Engine uses the planned field footprints and years to find and process Sentinel-2 imagery. With `submit_only = True`, it can continue processing after submission finishes and the laptop is closed. On a later run, `submit_only = False` retrieves completed results, caches selected-pixel tables as Parquet and assembles annual TIFFs. Matching verified work can be reused. [Transport and caching](../../docs/reference/transport.md)

In [ ]:
project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


## 6. Review collection status

Use the batch status to see which work finished and which can resume later. Verify the parcels and windows that pass imagery checks through preprocessing.

In [ ]:
# Status comes from this run's collection response.
show_mapping({"Collection status": result["status"].replace("_", " ").title()},
             title="Collection result")
show_table([
    {"Batch": entry["batch"], "Status": entry["result"]["status"].replace("_", " ").title()}
    for entry in result.get("batches", [])
], title="Collection status by batch")

## Saved outputs

Browse reference files, batch states and imagery through the organized collection index.

In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")
print("Next: data_collection/08_dataset_summary.ipynb; assess supplemental usable windows separately.")


Continue to [Notebook 08](08_dataset_summary.ipynb) for the combined dataset profile and [Preprocessing 01](../preprocessing/01_imagery_quality.ipynb) for imagery quality and coverage.